# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the FAIR^2 dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Explore metadata
print("Dataset loaded.")
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Keywords: {dataset.metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List available record sets and their @id
record_sets = dataset.record_sets
print('Available Record Sets:')
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}")
    print(f"  Description: {rs.description}")
    # List fields
    print("  Fields:")
    for f in rs.fields:
        print(f"    - {f.name} (@id: {f.id}, dataType: {f.data_type})")
    print("")

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis. All entities are referenced by their `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}
# List of record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show columns and sample data for the first record set
first_record_set_id = record_set_ids[0]
print(f"Columns for record set '{first_record_set_id}':")
print(dataframes[first_record_set_id].columns.tolist())

print(f"Sample records from '{first_record_set_id}':")
display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We apply common processing steps such as filtering numeric records, normalizing them, and grouping by key categorical fields.

Entities are referenced by their fields' `@id`.

In [ ]:
# Choose a record set and numeric/categorical fields by their @id
# For demonstration, use the first record set and display available fields
record_set_id = first_record_set_id  # Using the first record set
df = dataframes[record_set_id]
print(f"Field @ids for '{record_set_id}':")
print(df.columns.tolist())

# Guess numeric and group fields based on their names
# Let's assume field ids containing 'log_likelihood' or 'coefficient' are numeric, and 'ward' or 'gender' are group fields
numeric_field_ids = [col for col in df.columns if 'log_likelihood' in col or 'coefficient' in col or 'standard_error' in col or 'p_value' in col]
group_field_ids = [col for col in df.columns if 'ward' in col or 'gender' in col or 'county' in col]

if numeric_field_ids:
    numeric_field = numeric_field_ids[0]
    print(f"Using numeric field: {numeric_field}")
else:
    numeric_field = None
    print("No obvious numeric field found.")

if group_field_ids:
    group_field = group_field_ids[0]
    print(f"Using group field: {group_field}")
else:
    group_field = None
    print("No obvious group field found.")

# Filter records where numeric field > threshold (if numeric field found)
if numeric_field:
    threshold = 0  # Example threshold
    if pd.api.types.is_numeric_dtype(df[numeric_field]):
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with '{numeric_field}' > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by the group field and show means
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by '{group_field}':")
            display(grouped_df.head())
    else:
        print(f"Field '{numeric_field}' is not numeric. Skipping filtering and normalization.")
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize numeric data distributions and relationships between fields.

Use matplotlib and seaborn for basic visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of '{numeric_field}' in '{record_set_id}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If a group field is available, show boxplot
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"'{numeric_field}' by '{group_field}' in '{record_set_id}'")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the FAIR^2 dataset.

- The dataset provides detailed ordered logistic regression outputs relating to predictors of knowledge adoption and rangeland management.
- Exploratory analysis enables filtering, normalization, and grouping of variables identified by `@id`.
- Visualizations show distributions and group differences for key numeric fields.
- The FAIR^2 dataset is designed for transparent, reproducible research, and supports policy analysis and academic work on climate adaptation, extension services, and gender inclusion.
